### Structured output

Models can be requested to provide their response in a format matching a given schema. this is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods types and methods for enforcing structured output 

### Pydntic

pydantic models provide the richest feature set with field validation,description, and nested structures.

In [1]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x111ddd7f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x111dde510>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The rating of the movie out of 10")

In [ ]:
model.with_structured_output(Movie).invoke("Please provide details of the movie Inception") 
# can use include_raw = True to get the raw response from the model which 
# includes the structured output and other metadata inside the invoke response object.

Movie(title='Inception', director='Christopher Nolan', year=2010, rating=8.5)

### Nested Structure 

In [8]:
from pydantic import BaseModel, Field                                                                                             
   
class Car(BaseModel):                                                                                                             
      name: str = Field(description="The name of the car")
      year: int = Field(description="The year the car was manufactured")
      price: float = Field(description="The price of the car in USD")
                                                                                                                                    
class Brand(BaseModel):                                                                                                           
      name: str = Field(description="The name of the brand")                                                                        
      country: str = Field(description="The country of origin of the brand")                                                        
      model:list[Car] = Field(description="The model of the car")

  # Outside the class                                                                                                               
response = model.with_structured_output(Brand).invoke("Please provide details of the brand BMW and its models with V-8 engine")
print(response)

name='BMW' country='Germany' model=[Car(name='M5', year=2022, price=102000.0), Car(name='M850i', year=2022, price=111000.0), Car(name='X5 xDrive50i', year=2022, price=82000.0)]


### TypeDict

When to use TypedDict:
  - Simple dict structure hints                                                                                                     
  - When you don't need validation
  - Faster performance matters                                                                                                      
  - LangChain structured output (some models prefer it)                                                                             
                                                                                                                                    
  When to use Pydantic:                                                                                                             
  - Need real data validation                                                                                                       
  - Want type conversion                                                                                                            
  - Complex nested structures                                                                                                       
  - LangChain structured output with validation

In [ ]:
from typing import TypedDict, List
                                                                                                                                    
class SocialMedia(TypedDict):
      platform: str          
      followers: str
                     
class Celebrity(TypedDict):
      name: str                                                                                                                     
      profession: str                
      country: str                                                                                                                  
      age: int                                                                                                                      
      social_media: List[SocialMedia]                                                                                               
                                                                                                                                    
response = model.with_structured_output(Celebrity).invoke("Give me details about Cristiano Ronaldo including his profession, country, age, and his social media presence with follower counts")                                              
print(response)
print(response["name"])                                                                                                           
print(response["social_media"][0]["followers"])                                                                                                            

{'age': 38, 'country': 'Portugal', 'name': 'Cristiano Ronaldo', 'profession': 'Footballer', 'social_media': [{'followers': '245 million', 'platform': 'Instagram'}, {'followers': '75 million', 'platform': 'Twitter'}, {'followers': '160 million', 'platform': 'Facebook'}]}
Cristiano Ronaldo
245 million


In [13]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

### DataClasses

A dataclass is a class typically containing mainly data, although there aren't any restrictions. you create it using the @dataclass decorator
